In [1]:

%%writefile pp1_motor_engine.py
import numpy as np
from scipy.signal import butter, sosfiltfilt, find_peaks

# DEMO ONLY: arbitrary software-testing thresholds.
# NOT calibrated against real devices or clinical ratings.
DEMO_THRESHOLDS = {
    "TS": [0.12, 0.24, 0.40, 0.70],
    "BS": [210, 165, 120, 75],
    "RS": [2.4, 1.8, 1.2, 0.6],
    "GI": [110, 95, 80, 65]
}

DURATIONS = {"TS": 10, "BS": 10, "RS": 10, "GI": 180}


def check_sensor(data, seconds, fs, name):
    data = np.asarray(data, dtype=float)
    expected = int(round(seconds * fs))

    if data.shape != (expected, 3):
        raise ValueError(
            f"{name}: expected ({expected}, 3) samples"
        )
    if not np.isfinite(data).all():
        raise ValueError(f"{name}: invalid sensor values")

    return data


def extract_features(ts_acc, bs_taps, rs_gyro, gi_acc, fs=100):

    if not np.isfinite(fs) or fs <= 6:
        raise ValueError("Invalid sampling rate")

    ts = check_sensor(ts_acc, 10, fs, "TS")
    rs = check_sensor(rs_gyro, 10, fs, "RS")
    gi = check_sensor(gi_acc, 180, fs, "GI")

    taps = np.asarray(bs_taps, dtype=float)

    if taps.ndim != 1 or not np.isfinite(taps).all():
        raise ValueError("Invalid BS tap timestamps")

    if (np.any(taps < 0) or np.any(taps >= 10)
            or np.any(np.diff(taps) <= 0)):
        raise ValueError("Invalid BS tap timing")

    # TS: dynamic acceleration RMS
    ts_magnitude = np.linalg.norm(ts, axis=1)
    ts_dynamic = ts_magnitude - 9.81
    ts_rms = float(np.sqrt(np.mean(ts_dynamic**2)))

    # BS: capacitive touch tapping
    tap_rate = float(len(taps) * 60 / 10)
    tap_std = (
        float(np.std(np.diff(taps)))
        if len(taps) > 1 else None
    )

    # RS: active wrist movement speed proxy
    omega = np.linalg.norm(rs, axis=1)
    peaks, _ = find_peaks(
        omega,
        distance=max(1, int(0.45 * fs)),
        prominence=0.2
    )

    rs_peak = (
        float(np.mean(omega[peaks]))
        if len(peaks) else None
    )

    # GI: candidate steps from 180-second wrist recording
    magnitude = np.linalg.norm(gi, axis=1)

    sos = butter(
        3, [0.6, 3.0],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    filtered = sosfiltfilt(sos, magnitude)

    steps, _ = find_peaks(
        filtered,
        distance=max(1, int(0.4 * fs)),
        prominence=0.3
    )

    step_times = steps / fs
    cadence = float(len(steps) * 60 / 180)

    step_std = (
        float(np.std(np.diff(step_times)))
        if len(steps) > 1 else None
    )

    return {
        "TS": {
            "tremor_rms_ms2": ts_rms
        },
        "BS": {
            "tap_count": int(len(taps)),
            "tap_rate_per_min": tap_rate,
            "tap_interval_std_s": tap_std
        },
        "RS": {
            "mean_gyro_peak_rad_s": rs_peak,
            "movement_peaks": int(len(peaks))
        },
        "GI": {
            "step_candidates": int(len(steps)),
            "cadence_steps_min": cadence,
            "step_interval_std_s": step_std
        }
    }


def score_high(value, thresholds):
    return int(sum(value >= x for x in thresholds))


def score_low(value, thresholds):
    return int(sum(value < x for x in thresholds))


def assess_motor(ts_acc, bs_taps, rs_gyro, gi_acc,
                 fs=100, mode="demo"):

    if mode not in ("demo", "real"):
        raise ValueError("Use mode='demo' or mode='real'")

    features = extract_features(
        ts_acc, bs_taps, rs_gyro, gi_acc, fs
    )

    # Real-device readings are NOT clinically scored
    # until thresholds are properly calibrated.
    if mode == "real":
        return {
            "mode": "real",
            "features": features,
            "scores": None,
            "mps": None,
            "status": "UNCALIBRATED - FEATURES ONLY"
        }

    ts = score_high(
        features["TS"]["tremor_rms_ms2"],
        DEMO_THRESHOLDS["TS"]
    )
    bs = score_low(
        features["BS"]["tap_rate_per_min"],
        DEMO_THRESHOLDS["BS"]
    )

    rs_value = features["RS"]["mean_gyro_peak_rad_s"]
    rs = (
        score_low(rs_value, DEMO_THRESHOLDS["RS"])
        if rs_value is not None else None
    )

    gi = score_low(
        features["GI"]["cadence_steps_min"],
        DEMO_THRESHOLDS["GI"]
    )

    scores = {"TS": ts, "BS": bs, "RS": rs, "GI": gi}

    mps = (
        sum(scores.values()) / 16.0
        if all(v is not None for v in scores.values())
        else None
    )

    return {
        "mode": "demo",
        "features": features,
        "scores": scores,
        "mps": mps,
        "status": "DEMO ONLY - NOT CLINICALLY VALIDATED"
    }


Writing pp1_motor_engine.py


In [2]:

import numpy as np
import json
from pp1_motor_engine import assess_motor

fs = 100

# TS - 10 seconds
t = np.arange(0, 10, 1/fs)

ts_acc = np.column_stack([
    0.20 * np.sin(2*np.pi*5*t),
    0.10 * np.cos(2*np.pi*5*t),
    9.81 + 0.15 * np.sin(2*np.pi*5*t)
])

# BS - 10 seconds
bs_taps = np.arange(0.3, 10, 0.48)

# RS - 10 seconds
rs_gyro = np.column_stack([
    1.8 * np.sin(2*np.pi*1.2*t),
    0.10 * np.cos(2*np.pi*1.2*t),
    0.05 * np.sin(2*np.pi*1.2*t)
])

# GI - 180 seconds
tg = np.arange(0, 180, 1/fs)
rng = np.random.default_rng(42)

gi_acc = np.column_stack([
    0.15 * np.sin(2*np.pi*1.6*tg),
    0.10 * np.cos(2*np.pi*1.6*tg),
    9.81 + 1.2*np.sin(2*np.pi*1.6*tg)
    + 0.05*rng.standard_normal(len(tg))
])

# Run DEMO mode
demo = assess_motor(
    ts_acc, bs_taps, rs_gyro, gi_acc,
    fs=100,
    mode="demo"
)

print("PP1 MOTOR SCORING ENGINE")
print("------------------------------")
print("Mode:", demo["mode"])

for name, value in demo["scores"].items():
    print(f"{name}: {value}/4")

print("Temporary MPS:", demo["mps"])
print("Status:", demo["status"])

# Test REAL mode - features only
real = assess_motor(
    ts_acc, bs_taps, rs_gyro, gi_acc,
    fs=100,
    mode="real"
)

assert real["scores"] is None
assert real["mps"] is None

print("\nREAL MODE SAFETY CHECK")
print("Scores:", real["scores"])
print("MPS:", real["mps"])
print("Status:", real["status"])

# Save machine-readable demo result
with open("pp1_demo_result.json", "w") as f:
    json.dump(demo, f, indent=2)

print("\nResult saved: pp1_demo_result.json")


PP1 MOTOR SCORING ENGINE
------------------------------
Mode: demo
TS: 0/4
BS: 2/4
RS: 1/4
GI: 1/4
Temporary MPS: 0.25
Status: DEMO ONLY - NOT CLINICALLY VALIDATED

REAL MODE SAFETY CHECK
Scores: None
MPS: None
Status: UNCALIBRATED - FEATURES ONLY

Result saved: pp1_demo_result.json


In [3]:
%pip -q install gradio


In [4]:

import gradio as gr
import numpy as np
from pp1_motor_engine import assess_motor

def run_dashboard(tremor, tap_gap, wrist_speed, gait_hz, mode):

    fs = 100

    # TS, BS, RS: 10-second demonstrations
    t = np.arange(0, 10, 1/fs)

    ts_acc = np.column_stack([
        tremor * np.sin(2*np.pi*5*t),
        tremor * 0.5 * np.cos(2*np.pi*5*t),
        9.81 + tremor * 0.75 * np.sin(2*np.pi*5*t)
    ])

    bs_taps = np.arange(0.3, 10, tap_gap)

    rs_gyro = np.column_stack([
        wrist_speed * np.sin(2*np.pi*1.2*t),
        0.10 * np.cos(2*np.pi*1.2*t),
        0.05 * np.sin(2*np.pi*1.2*t)
    ])

    # GI: full 180-second demonstration
    tg = np.arange(0, 180, 1/fs)
    rng = np.random.default_rng(42)

    gi_acc = np.column_stack([
        0.15 * np.sin(2*np.pi*gait_hz*tg),
        0.10 * np.cos(2*np.pi*gait_hz*tg),
        9.81
        + 1.2 * np.sin(2*np.pi*gait_hz*tg)
        + 0.05 * rng.standard_normal(len(tg))
    ])

    engine_mode = (
        "demo" if mode == "Demo scoring" else "real"
    )

    result = assess_motor(
        ts_acc, bs_taps, rs_gyro, gi_acc,
        fs=fs,
        mode=engine_mode
    )

    if result["scores"] is None:
        report = """
### Real-Mode Safety Check

**Scores:** Not available

**MPS:** Not available

**Status:** Uncalibrated — features only

This test still uses simulated inputs. No physical device is connected.
"""
    else:
        s = result["scores"]

        report = f"""
### Motor Assessment Results — DEMO ONLY

| Motor Test | Automatic Score |
|---|---:|
| TS – Tremor | {s['TS']}/4 |
| BS – Bradykinesia | {s['BS']}/4 |
| RS – Motor Mobility Proxy | {s['RS']}/4 |
| GI – Gait | {s['GI']}/4 |

### Temporary MPS: {result['mps'] if result['mps'] is not None else 'N/A'}

**Status:** Uncalibrated demonstration. Not a diagnosis or PD probability.
"""

    return report, result["features"]


with gr.Blocks(title="PP1 Motor Assessment") as app:

    gr.Markdown(
        "# Portable Motor Assessment AI — PP1\n"
        "**Interactive Demo Dashboard**\n\n"
        "Simulated sensor signals only. "
        "Scoring thresholds are not clinically validated."
    )

    mode = gr.Radio(
        ["Demo scoring", "Real-mode gate test"],
        value="Demo scoring",
        label="Assessment Mode"
    )

    with gr.Row():
        tremor = gr.Slider(
            0.05, 1.5,
            value=0.2, step=0.05,
            label="TS – Tremor Signal Amplitude"
        )

        tap_gap = gr.Slider(
            0.2, 1.2,
            value=0.48, step=0.02,
            label="BS – Seconds Between Taps"
        )

    with gr.Row():
        wrist_speed = gr.Slider(
            0.2, 3.5,
            value=1.8, step=0.1,
            label="RS – Simulated Wrist Movement Speed"
        )

        gait_hz = gr.Slider(
            0.8, 2.0,
            value=1.6, step=0.1,
            label="GI – Simulated Gait Frequency (Hz)"
        )

    run_button = gr.Button(
        "Run Motor Assessment",
        variant="primary"
    )

    results = gr.Markdown(
        "Click **Run Motor Assessment** to begin."
    )

    extracted = gr.JSON(
        label="Extracted Sensor Features"
    )

    run_button.click(
        fn=run_dashboard,
        inputs=[
            tremor, tap_gap,
            wrist_speed, gait_hz, mode
        ],
        outputs=[results, extracted]
    )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ec8d1ace9c58c23c7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:

import numpy as np
import pandas as pd
from pathlib import Path
from pp1_motor_engine import assess_motor

folder = Path("/content/PP1_Sensor_CSV_Test")
folder.mkdir(exist_ok=True)

fs = 100

# Use the simulated signals from our previous test cell
# ts_acc, rs_gyro, gi_acc, bs_taps

def save_sensor_csv(filename, data, columns, fs):
    data = np.asarray(data)
    timestamps = np.arange(len(data)) / fs

    df = pd.DataFrame(data, columns=columns)
    df.insert(0, "time_s", timestamps)
    df.to_csv(folder / filename, index=False)

# 1. Export the four test recordings
save_sensor_csv(
    "TS_10s.csv",
    ts_acc,
    ["ax_ms2", "ay_ms2", "az_ms2"], fs
)

save_sensor_csv(
    "RS_10s.csv",
    rs_gyro,
    ["gx_rad_s", "gy_rad_s", "gz_rad_s"], fs
)

save_sensor_csv(
    "GI_180s.csv",
    gi_acc,
    ["ax_ms2", "ay_ms2", "az_ms2"], fs
)

pd.DataFrame({
    "tap_time_s": bs_taps
}).to_csv(folder / "BS_10s.csv", index=False)

print("FOUR SENSOR CSV FILES CREATED")

# 2. Read the CSV recordings back
def load_sensor_csv(filename, columns, duration):
    df = pd.read_csv(folder / filename)

    expected_columns = ["time_s"] + columns
    if list(df.columns) != expected_columns:
        raise ValueError(f"Wrong columns: {filename}")

    expected_samples = duration * fs
    if len(df) != expected_samples:
        raise ValueError(f"Wrong duration: {filename}")

    expected_times = np.arange(expected_samples) / fs
    if not np.allclose(
        df["time_s"].to_numpy(),
        expected_times,
        atol=0.0005,
        rtol=0
    ):
        raise ValueError(f"Invalid timestamps: {filename}")

    return df[columns].to_numpy(dtype=float)

acc_columns = ["ax_ms2", "ay_ms2", "az_ms2"]
gyro_columns = ["gx_rad_s", "gy_rad_s", "gz_rad_s"]

ts_loaded = load_sensor_csv(
    "TS_10s.csv", acc_columns, 10
)

rs_loaded = load_sensor_csv(
    "RS_10s.csv", gyro_columns, 10
)

gi_loaded = load_sensor_csv(
    "GI_180s.csv", acc_columns, 180
)

bs_df = pd.read_csv(folder / "BS_10s.csv")
assert list(bs_df.columns) == ["tap_time_s"]
bs_loaded = bs_df["tap_time_s"].to_numpy(dtype=float)

# 3. Assess imported CSV data - DEMO MODE
demo_result = assess_motor(
    ts_loaded,
    bs_loaded,
    rs_loaded,
    gi_loaded,
    fs=100,
    mode="demo"
)

print("\nCSV IMPORT - DEMO RESULTS")
print("Scores:", demo_result["scores"])
print("Temporary MPS:", demo_result["mps"])

# 4. Real mode refuses uncalibrated scores
real_result = assess_motor(
    ts_loaded,
    bs_loaded,
    rs_loaded,
    gi_loaded,
    fs=100,
    mode="real"
)

assert real_result["scores"] is None
assert real_result["mps"] is None

print("\nREAL MODE SAFETY CHECK")
print("Scores:", real_result["scores"])
print("MPS:", real_result["mps"])
print("Status:", real_result["status"])

print("\nCSV SENSOR INTEGRATION TEST COMPLETED")


FOUR SENSOR CSV FILES CREATED

CSV IMPORT - DEMO RESULTS
Scores: {'TS': 0, 'BS': 2, 'RS': 1, 'GI': 1}
Temporary MPS: 0.25

REAL MODE SAFETY CHECK
Scores: None
MPS: None
Status: UNCALIBRATED - FEATURES ONLY

CSV SENSOR INTEGRATION TEST COMPLETED


In [6]:

import gradio as gr
import pandas as pd
import numpy as np
from pathlib import Path
from pp1_motor_engine import assess_motor

FS = 100
DEMO_FOLDER = Path("/content/PP1_Sensor_CSV_Test")

ACC_COLS = ["ax_ms2", "ay_ms2", "az_ms2"]
GYRO_COLS = ["gx_rad_s", "gy_rad_s", "gz_rad_s"]


def read_xyz_csv(path, columns, duration):
    if not path:
        raise ValueError("Please provide all four CSV files.")

    df = pd.read_csv(path)
    expected_cols = ["time_s"] + columns

    if list(df.columns) != expected_cols:
        raise ValueError(
            f"Expected columns: {expected_cols}"
        )

    expected_rows = duration * FS

    if len(df) != expected_rows:
        raise ValueError(
            f"Expected {expected_rows} rows, "
            f"but found {len(df)}"
        )

    values = df.to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError("CSV contains invalid values.")

    expected_time = np.arange(expected_rows) / FS

    if not np.allclose(
        values[:, 0],
        expected_time,
        atol=0.0005,
        rtol=0
    ):
        raise ValueError(
            "Timestamp or sampling rate mismatch."
        )

    return values[:, 1:]


def read_bs_csv(path):
    if not path:
        raise ValueError("BS CSV is missing.")

    df = pd.read_csv(path)

    if list(df.columns) != ["tap_time_s"]:
        raise ValueError(
            "BS CSV requires tap_time_s column."
        )

    return df["tap_time_s"].to_numpy(dtype=float)


def assess_csv_files(ts_path, bs_path, rs_path,
                     gi_path, mode):

    try:
        ts = read_xyz_csv(ts_path, ACC_COLS, 10)
        bs = read_bs_csv(bs_path)
        rs = read_xyz_csv(rs_path, GYRO_COLS, 10)
        gi = read_xyz_csv(gi_path, ACC_COLS, 180)

        engine_mode = (
            "demo" if mode == "Demo scoring" else "real"
        )

        result = assess_motor(
            ts, bs, rs, gi,
            fs=FS,
            mode=engine_mode
        )

        if result["scores"] is None:
            report = """
## CSV Assessment — Features Only

**TS, BS, RS, GI Scores:** Unavailable

**MPS:** Unavailable

**Status:** UNCALIBRATED

CSV files were processed, but no clinical
scoring thresholds have been validated.
"""
        else:
            s = result["scores"]
            mps = result["mps"]

            mps_text = (
                f"{mps:.4f}"
                if mps is not None else "N/A"
            )

            report = f"""
## Automatic Motor Assessment — DEMO ONLY

| Test | Score |
|---|---:|
| TS – Tremor | {s['TS']}/4 |
| BS – Bradykinesia | {s['BS']}/4 |
| RS – Motor Mobility Proxy | {s['RS'] if s['RS'] is not None else 'N/A'}/4 |
| GI – Gait | {s['GI']}/4 |

### Temporary MPS: {mps_text}

**Status:** Uncalibrated demonstration.

Not a PD diagnosis or validated probability.
"""

        return report, result["features"]

    except Exception as error:
        return f"### CSV Error\n\n{str(error)}", {}


def use_saved_demo(mode):
    return assess_csv_files(
        str(DEMO_FOLDER / "TS_10s.csv"),
        str(DEMO_FOLDER / "BS_10s.csv"),
        str(DEMO_FOLDER / "RS_10s.csv"),
        str(DEMO_FOLDER / "GI_180s.csv"),
        mode
    )


with gr.Blocks(title="PP1 CSV Motor Assessment") as csv_app:

    gr.Markdown("""
# PP1 — Sensor CSV Assessment

Upload four sensor files or use saved demo CSVs.

**Fixed test durations:**
TS 10s | BS 10s | RS 10s | GI 180s

**Required units:** acceleration m/s²,
gyroscope rad/s, sampling rate 100 Hz.

Demo scores are not clinically validated.
""")

    mode = gr.Radio(
        ["Demo scoring", "Real-mode gate test"],
        value="Demo scoring",
        label="Processing Mode"
    )

    with gr.Row():
        ts_file = gr.File(
            label="TS — 10s CSV",
            file_types=[".csv"],
            type="filepath"
        )

        bs_file = gr.File(
            label="BS — 10s CSV",
            file_types=[".csv"],
            type="filepath"
        )

    with gr.Row():
        rs_file = gr.File(
            label="RS — 10s CSV",
            file_types=[".csv"],
            type="filepath"
        )

        gi_file = gr.File(
            label="GI — 180s CSV",
            file_types=[".csv"],
            type="filepath"
        )

    process_button = gr.Button(
        "Process Uploaded CSV Files",
        variant="primary"
    )

    demo_button = gr.Button(
        "Use Existing Colab Demo CSVs"
    )

    report = gr.Markdown()
    extracted_features = gr.JSON(
        label="Extracted Features"
    )

    process_button.click(
        fn=assess_csv_files,
        inputs=[
            ts_file, bs_file,
            rs_file, gi_file, mode
        ],
        outputs=[report, extracted_features]
    )

    demo_button.click(
        fn=use_saved_demo,
        inputs=[mode],
        outputs=[report, extracted_features]
    )

csv_app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fdf889d626da562130.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
from pathlib import Path
import shutil
import sys

root = Path("/content/Parkinson_Motor_AI")
target = root / "PP1" / "pp1_motor_engine.py"

# Find uploaded source file
candidates = [
    target,
    Path("/content/pp1_motor_engine.py")
]

source = next((p for p in candidates if p.is_file()), None)

assert source is not None, "Upload pp1_motor_engine.py first!"

target.parent.mkdir(parents=True, exist_ok=True)

if source.resolve() != target.resolve():
    shutil.copy2(source, target)

sys.path.insert(0, str(target.parent))

from pp1_motor_engine import assess_motor, DURATIONS

assert DURATIONS == {
    "TS": 10,
    "BS": 10,
    "RS": 10,
    "GI": 180
}

print("PP1 MOTOR ENGINE IMPORT SUCCESSFUL!")
print("Engine:", target)
print("Test durations:", DURATIONS)
print("Assessment function: READY")

PP1 MOTOR ENGINE IMPORT SUCCESSFUL!
Engine: /content/Parkinson_Motor_AI/PP1/pp1_motor_engine.py
Test durations: {'TS': 10, 'BS': 10, 'RS': 10, 'GI': 180}
Assessment function: READY
